<a href="https://colab.research.google.com/github/yamazaki-riko/python_learning/blob/koshien_google-colab/koshien_bt_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# セル1：ライブラリの準備
!pip install scipy -q

import sqlite3
import pandas as pd
import numpy as np
from scipy.optimize import minimize

# DBに接続
from google.colab import drive
drive.mount('/content/drive')
conn = sqlite3.connect("/content/drive/MyDrive/koshien/koshien.db")
print("✅ DB接続完了")

Mounted at /content/drive
✅ DB接続完了


In [2]:
# セル2：試合データを読み込む
df_games = pd.read_sql("""
    SELECT year, winner_school, loser_school
    FROM tmp_tournament_games
    WHERE winner_score IS NOT NULL  -- 不戦勝を除外
""", conn)

print(f"試合数: {len(df_games)}件")
print(f"対象年: {sorted(df_games['year'].unique())}")
display(df_games.head())

試合数: 207件
対象年: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,year,winner_school,loser_school
0,2022,国学院栃木,日大三島
1,2022,明豊,樹徳
2,2022,一関学院,京都国際
3,2022,八戸学院光星,愛工大名電
4,2022,愛工大名電,星稜


In [3]:
# セル3：Bradley-Terry モデルを計算する
def calc_bradley_terry(df):
    """
    試合結果からBradley-Terryの強さパラメータを推定する
    df: winner_school, loser_school の列を持つDataFrame
    """

    # 全チームのリストを作る
    teams = sorted(set(df["winner_school"]) | set(df["loser_school"]))
    idx   = {t: i for i, t in enumerate(teams)}
    n     = len(teams)

    # 負の対数尤度（これを最小化することで強さを推定する）
    def neg_log_likelihood(params):
        strength = np.exp(params)  # 強さは必ず正の値
        total = 0.0
        for _, row in df.iterrows():
            w = idx[row["winner_school"]]
            l = idx[row["loser_school"]]
            # 勝者が勝つ確率の対数を足していく
            total -= np.log(strength[w] / (strength[w] + strength[l]))
        return total

    # 最適化（全チームの強さを同時に推定）
    result = minimize(
        neg_log_likelihood,
        x0     = np.zeros(n),   # 初期値は全員同じ強さ
        method = "L-BFGS-B"
    )

    strength = np.exp(result.x)

    df_result = pd.DataFrame({
        "school":   teams,
        "strength": strength
    }).sort_values("strength", ascending=False).reset_index(drop=True)

    df_result["rank"]   = df_result.index + 1
    df_result["points"] = range(49, 49 - len(df_result), -1)

    return df_result


# 全年データでモデルを計算
df_bt = calc_bradley_terry(df_games)

print(f"✅ 計算完了：{len(df_bt)}校")
display(df_bt.head(20))

✅ 計算完了：135校


,school,strength,rank,points
0,慶應義塾,2.326292e+48,1,49
1,沖縄尚学,5.777375e+38,2,48
2,土浦日大,1.659725e+34,3,47
3,仙台育英,6.926763e+32,4,46
4,早稲田実,4.622940e+27,5,45
5,下関国際,3.481315e+26,6,44
6,愛工大名電,1.578524e+25,7,43
7,八戸学院光星,1.578519e+25,8,42
8,市和歌山,6.935732e+22,9,41
9,中京大中京,2.986885e+22,10,40


In [4]:
# セル4：2025年の出場校だけに絞る
# 2025年の出場校リストを取得
df_2025 = pd.read_sql("""
    SELECT school_name, district, summer_appearances
    FROM tmp_koshien_appearances_summer
    WHERE year = 2025
""", conn)

# BTモデルの結果と結合
df_final = df_2025.merge(
    df_bt[["school", "strength", "rank", "points"]],
    left_on  = "school_name",
    right_on = "school",
    how      = "left"
)

# 強さが計算できた学校だけ順位を付け直す
df_final = df_final.sort_values("strength", ascending=False).reset_index(drop=True)
df_final["bt_rank"]   = df_final.index + 1
df_final["bt_points"] = range(49, 0, -1)

print("🏆 2025年 甲子園 BTモデル予測ランキング")
display(df_final[["bt_rank","school_name","district","strength","bt_points"]].head(20))

🏆 2025年 甲子園 BTモデル予測ランキング


,bt_rank,school_name,district,strength,bt_points
0,1,沖縄尚学,沖縄,5.777375e+38,49
1,2,仙台育英,宮城,6.926763e+32,48
2,3,大阪桐蔭,大阪,4.587831e+16,47
3,4,聖光学院,福島,1.127395e+09,46
4,5,山梨学院,山梨,8.429851e+08,45
5,6,京都国際,京都,4.990355e+08,44
6,7,日大山形,山形,3.911438e+08,43
7,8,横浜,西東京,3.876265e+08,42
8,9,東海大相模,神奈川,3.416891e+08,41
9,10,神村学園,鹿児島,2.875495e+08,40


In [5]:
# 強さを 0〜100 のスコアに正規化する
df_final["strength_log"] = np.log(df_final["strength"].replace(0, np.nan))

s_min = df_final["strength_log"].min()
s_max = df_final["strength_log"].max()

df_final["score"] = (
    (df_final["strength_log"] - s_min) / (s_max - s_min) * 100
).round(1)

# 表示
print("🏆 2025年 甲子園 BTモデル予測ランキング（スコア正規化済み）")
display(df_final[["bt_rank","school_name","district","score","bt_points"]])

🏆 2025年 甲子園 BTモデル予測ランキング（スコア正規化済み）


,bt_rank,school_name,district,score,bt_points
0,1,沖縄尚学,沖縄,100.0,49
1,2,仙台育英,宮城,90.2,48
2,3,大阪桐蔭,大阪,63.5,47
3,4,聖光学院,福島,50.9,46
4,5,山梨学院,山梨,50.7,45
5,6,京都国際,京都,50.3,44
6,7,日大山形,山形,50.1,43
7,8,横浜,西東京,50.1,42
8,9,東海大相模,神奈川,50.0,41
9,10,神村学園,鹿児島,49.9,40
